In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = '<YOUR_API_KEY>'
BASE_URL = "https://www.googleapis.com/youtube/v3"

In [3]:
upload_playlist_df= pd.read_csv('../upload_playlist.csv')

In [4]:
video_ids = upload_playlist_df[['video_id']]

In [5]:
video_ids.columns
len(video_ids)

31358

In [6]:
endpoint = "videos"
videos_url = f"{BASE_URL}/{endpoint}"

all_video_data=[]

for batch in range(0,len(video_ids),50):
    params = {
        "key" : API_KEY,
        "part" : "snippet,contentDetails,status,statistics",
        "id" : ','.join(video_ids.iloc[batch:batch+50]['video_id'])
    }
    response = requests.get(videos_url, params=params)
    all_video_data.extend(response.json().get('items',[]))

In [7]:
len(all_video_data)

31358

In [8]:
# Removing data for video ids not having 'processed' status
video_id_with_no_processed_status=[]
for data in all_video_data:
    if data['status']['uploadStatus']!='processed':
        video_id_with_no_processed_status.append(data['id'])
video_id_with_no_processed_status

['iYt3c_X5bLs',
 'ySSRi5dfws4',
 '1dXIWxkLy3E',
 'gknWRtdegik',
 '5HkbMytJ5Uw',
 'q5Fa6sTo6W0',
 'J9mxt1XD_S0',
 'iipR5yUp36o',
 'ZiChWM8lTho',
 'ucffZfKJ7gU',
 'jEK8Yw-SVPw',
 'DHb2lpSt8NM',
 'NUT_XMbUSSY',
 'XRb9c2XjZds']

In [9]:
videos_df = pd.DataFrame({
                    "video_id": [data['id'] for data in all_video_data],
                    "channel_id": [data['snippet']['channelId'] for data in all_video_data],
                    "video-title": [data['snippet']['title'] for data in all_video_data],
                    "video_description": [data['snippet']['description'] for data in all_video_data],
                    "publish_date": [data['snippet']['publishedAt'] for data in all_video_data],
                    "tags": [data['snippet'].get('tags') for data in all_video_data],
                    "category_id": [data['snippet'].get('categoryId') for data in all_video_data],
                    "duration": [data['contentDetails'].get('duration') for data in all_video_data],
                    "dimension": [data['contentDetails'].get('dimension') for data in all_video_data],
                    "caption_enabled" : [data['contentDetails'].get('caption') for data in all_video_data],
                    "view_count": [data['statistics'].get('viewCount') for data in all_video_data],
                    "like_count": [data['statistics'].get('likeCount') for data in all_video_data],
                    "dislike_count": [data['statistics'].get('dislikeCount') for data in all_video_data],
                    "favourite_count": [data['statistics'].get('favoriteCount') for data in all_video_data],
                    "comment_count": [data['statistics'].get('commentCount') for data in all_video_data],
                    }
)

In [10]:
final_df = videos_df[~videos_df['video_id'].isin(video_id_with_no_processed_status)]

In [11]:
final_df.to_csv('videos.csv')

DATA CLEANING NAME ERRORS FROM video_id

In [ ]:
upload_playlist_df
name_error_df = upload_playlist_df[upload_playlist_df['video_id']=='#NAME?']

In [ ]:
name_error_df

In [ ]:
import requests

endpoint = "playlistItems"
final_url = f"{BASE_URL}/{endpoint}"

def get_details(playlist_id):
    params = {
        "key": API_KEY,
        "part": "contentDetails,id,snippet",
        "playlistId": playlist_id,   # Correct parameter name
        "maxResults": 50             # Maximum allowed per request
    }

    all_data = []

    while True:
        response = requests.get(final_url, params=params)
        response.raise_for_status()

        data = response.json()
        all_data.extend(data.get("items", []))

        if "nextPageToken" in data:
            params["pageToken"] = data["nextPageToken"]   # Correct parameter name
        else:
            break

    return all_data

for playlist_id in name_error_df["upload_playlist_id"]:
    data = get_details(playlist_id)
    print(data)
    break